# 00 — Task Formulation and EDA

<details open>
<summary><strong>Project Story</strong></summary>

This notebook starts Team 10's final project for **Fundamentals of Natural Language / NLP-I** at Universitat Autonoma de Barcelona, academic year 2025-2026. Our team is Phoebe Iglesias (1713459), David Redrejo (1790336), and Pau Rossell (1750424). The project is supervised by Ernest Valveny and Lei Kang.

The competition is **UAB-ASHO AI Codification** (`uab-asho-ai-codification`). The task looks small at first: given a short clinical literal, predict one ICD category prefix called `y_category`. But this is exactly why it is interesting. Clinical coding turns messy medical language into structured categories that can be searched, audited, and analyzed. Even a two-word phrase can carry administrative and clinical meaning.

This first notebook also anchors the follow-up discussions with Lei Kang and the final presentation context: before we argue for TF-IDF, SVMs, RoBERTa, or ensembles, we need to prove that we understand the data and the annotations.
</details>

## <details open><summary>1. Why We Start With Data and Annotations</summary></details>

The target is not the full ICD code. The project objective is to predict exactly one category prefix:

```python
y_category = Code.astype(str).str[0]
```

There are expected to be 36 categories: digits `0`-`9` and letters `A`-`Z`. If the actual CSV files contain fewer, more, or different labels, this should not be hidden. It changes how we interpret the task and how fair our evaluation is.

So the first real technical phase is **Analyzing the data and the annotations**. No model is trained here.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import discover_csv_files, inspect_csv_file

csv_files = discover_csv_files(PROJECT_ROOT / 'data')
csv_files

## <details open><summary>2. Inventory of CSV Files</summary></details>

We inspect every CSV file under `data/`, not only the filenames we expect. This protects us from silently ignoring a renamed training file, a sample submission file, or an extra ICD dictionary.

In [ ]:
summaries = []
for path in csv_files:
    summary, df = inspect_csv_file(path)
    summaries.append(summary)
    print(f'\n=== {path.relative_to(PROJECT_ROOT)} ===')
    print('shape:', df.shape)
    print('columns:', df.columns.tolist())
    display(df.head())
    print('null counts:')
    print(df.isna().sum())
    print('duplicate rows:', int(df.duplicated().sum()))

if not csv_files:
    print('No CSV files found under data/. Place the Kaggle files in data/raw/ and rerun this notebook or the validation script.')

## <details open><summary>3. Annotation Contract</summary></details>

The contract we use for the rest of the project is:

- training data must provide `Code` and `Literal`,
- leaderboard/test data must provide at least `id` and `Literal`,
- if `y_category` is present in a leaderboard-like file, it is validated; if it is absent, we treat the file as unlabeled Kaggle test data,
- `y_category` is derived from the first character of `Code`,
- labels are sorted and mapped to integer ids using `label2id` and `id2label`.

The result of this validation is saved by the command-line script so the repository has a reproducible record.

In [ ]:
!python scripts/analyze_data_annotations.py

## <details open><summary>4. Result Interpretation</summary></details>

The generated files are:

- `outputs/eda/data_file_inventory.csv`
- `outputs/eda/schema_validation.json`
- `reports/tables/data_schema_summary.csv`

If the command reports missing data, that is already a useful finding: the project cannot honestly train or report final metrics until the raw Kaggle CSVs are restored. Once the files are present, this notebook becomes the first evidence table for the final report.